# VinaScreen
For screening multiple ligands

Requirements: 
1. Ligands: sdf, smi or pdb files
2. Receptor: PDB files of receptors in receptor folder
3. Parameters: config.txt of files specifying binding site, exhaustiveness and number of poses per ligand

*Important Note:* There is only one binding site coordinates that are recognized in the config.txt file. For screening multiple receptors with different binding site coordinates, you need to run this one receptor at a time.


## Prepare Receptor with Meeko

**Important**

Make sure your receptor has been cleaned of its co-crystallized ligand and water molecules. This script has an option to process cofactors in the receptor. Please double check on your research design if the cofactor is required to be present on your receptor, and carefully choose which jupyter cell you need to run

### Single Receptor Preparation

In [18]:
#Input files and define output filenames
inputpdb = 'humCYP51.pdb'    #input receptor file here with .pdb file name extension
output = 'humCYP51'          #type your desired output filename without extension

In [19]:
#fix missing sidechains and alernate structures
!pdbfixer {inputpdb} --output={output}_fix.pdb --add-atoms=heavy --keep-heterogens=all

In [20]:
#convert pdb to pdbqt with meeko. this is for proteins with heme cofactors
!mk_prepare_receptor.py -i {output}_fix.pdb -o {output}_heme -p

@> 3633 atoms and 1 coordinate set(s) were parsed in 0.02s.
/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/polymer.py:856: RuntimeWarning: Input residues {'A:601': 'HEM'} not in residue_templates

  warnings.warn(err, RuntimeWarning)
/home/nikka/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/polymer.py:857: RuntimeWarning: Trying to resolve unknown residues by building chemical templates... 
  warnings.warn("Trying to resolve unknown residues by building chemical templates... ", RuntimeWarning)
Molecule contains metal with unspecified charge state -> charging nonmetal coordinated atoms...
All metals will be neutralized and the nonmetal coordinated atoms will be charged according to their explicit valence. 
Total charge of the molecule after recharging: 2

Files written:
humCYP51_heme.pdbqt <-- static (i.e., rigid) receptor input file


In [10]:
# if the receptor has a heme, check the presence of heme group in meeko
check='humCYP51_heme.pdbqt'
with open(check, "r") as f:
    for line in f:
        if "HEM" in line and "FE" in line:
            print(line.strip())

ATOM   4418  FE  HEM A 601      19.144  -3.550  22.456  1.00  0.00     1.241 Fe


### Batch Receptor preparation

In [15]:
#This is for ensuring that the receptors are cleaned of ligand, water or any other non-protein residues. skip if cofactors are needed in the docking
#NOTE: This version does NOT align/superimpose structures to any reference, so pre-existing
#ensemble alignment on specific chains/residues is preserved.

import pandas as pd
from src.receptor_cleaner import batch_clean_ensemble

# ==========================================
# 1. PARAMETERS
# ==========================================
RAW_PDB_DIR       = "TM-receptors/"                       # Folder containing your crystal structures
CLEAN_PDB_DIR     = "clean_TM-conf"                  # Destination folder for cleaned structures
AUDIT_REPORT_PATH = "TM-receptor_cleaning_report.csv"      # CSV audit report path

TARGET_CHAINS     = ["A", "B", "C"]                         # Retains chains A/B/C (falls back safely if absent)

# ==========================================
# 2. RUN BATCH PREPARATION
# ==========================================
audit_df = batch_clean_ensemble(
    input_dir=RAW_PDB_DIR,
    output_dir=CLEAN_PDB_DIR,
    keep_chains=TARGET_CHAINS,
    audit_report_path=AUDIT_REPORT_PATH
)

# ==========================================
# 3. DISPLAY AUDIT SUMMARY
# ==========================================
display(audit_df[[
    "filename",
    "status",
    "missing_heavy_atoms_added",
    "nonstandard_replaced"
]])

[14:28:18] [INFO] Discovered 4 structure(s). Retaining: chains ['A', 'B', 'C'].
[14:28:18] [INFO] Processing: '5jmn.pdb'
[14:28:20] [INFO] Processing: '6zod.pdb'
[14:28:22] [INFO] Processing: 'TM2-5jmn.pdb'
[14:28:24] [INFO] Processing: 'TM3-6zod.pdb'
[14:28:26] [INFO] Audit log saved to '/home/amicon19/Docking/TM-receptor_cleaning_report.csv'.
[14:28:26] [INFO] =============================================
[14:28:26] [INFO] BATCH ENSEMBLE CLEANUP SUMMARY (no alignment performed)
[14:28:26] [INFO] =============================================
[14:28:26] [INFO] Total structures processed : 4
[14:28:26] [INFO] Successfully repaired      : 4
[14:28:26] [INFO] Cleaned PDB directory      : /home/amicon19/Docking/clean_TM-conf
[14:28:26] [INFO] =============================================


,filename,status,missing_heavy_atoms_added,nonstandard_replaced
0,5jmn.pdb,Success,3,0
1,6zod.pdb,Success,3,0
2,TM2-5jmn.pdb,Success,3,0
3,TM3-6zod.pdb,Success,3,0


In [16]:
# Use this cell for protein-only receptors
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.receptor_pdbqt import batch_convert_protein_only, ALLOWED_PROTONATION_TAGS

# ==========================================
# 1. PARAMETERS
# ==========================================
INPUT_PDB_DIR       = "clean_TM-conf/"   # Folder from upstream receptor_cleaner
OUTPUT_PDBQT_DIR    = "clean_TM-conf/"     # Output folder for .pdbqt files
AUDIT_REPORT_PATH   = "TM-receptor_pdbqt_summary.csv"  # Audit table path

# --- Tier 1: global pH (required) ---
GLOBAL_PH = 7.4   # pH used by PROPKA to titrate all standard residues

# --- Tier 2: manual protonation overrides (optional) ---
# Type "Yes" if one or more residues need a fixed protonation state that differs from what the global pH would assign (e.g. determined from crystallographic H-bond geometry at the binding site). 
# Otherwise type "No".
USE_CUSTOM_PROTONATION = "No"   # <-- type "Yes" or "No"

# If USE_CUSTOM_PROTONATION = "Yes", list every override residue here.
# Format: (chain_id, residue_number): "TAG"
# Allowed tags: 
#      ASH: Neutral Asp, protonated carboxylic acid;
#      GLH: Neutral Glu, protonated carboxylic acid;
#      HID: Neutral His, proton on the delta nitrogen (Nδ/HD1) of the imidazole ring;
#      HIE: Neutral His, proton on the epsilon nitrogen (Nε/HE2) of the imidazole ring  
#      HIP: Positively charged His, both ring nitrogens protonated
#      LYN: Neutral Lys, deprotonated amine (–NH2 instead of –NH3⁺)
#      CYM: Negatively charged Cys thiolate (–S⁻ instead of –SH)
#      TYM: Negatively charged Tyr phenolate (–O⁻ instead of –OH)
# You can add as many entries as needed; they all apply to every structure in the batch.

RESIDUE_OVERRIDES_INPUT = {
    ("A", 407): "ASP",   # e.g. catalytic His, epsilon-tautomer per H-bond geometry
    ("A", 408): "ASP",   # e.g. neutral Asp, carboxylate O-H points toward relay partner
    ("A", 940): "LYS",   # e.g. deprotonated Cys nucleophile
    ("A", 947): "GLU",
}

# ==========================================
# 2. RESOLVE PROTONATION SWITCH
# ==========================================
use_custom_flag = str(USE_CUSTOM_PROTONATION).strip().lower() in {"yes", "y", "true"}

if use_custom_flag:
    invalid = {k: v for k, v in RESIDUE_OVERRIDES_INPUT.items() if v.upper() not in ALLOWED_PROTONATION_TAGS}
    if invalid:
        raise ValueError(f"Invalid protonation tag(s) in RESIDUE_OVERRIDES_INPUT: {invalid}. "
                          f"Allowed tags: {sorted(ALLOWED_PROTONATION_TAGS)}")
    residue_overrides = RESIDUE_OVERRIDES_INPUT
    print(f"Tier 2 enabled: {len(residue_overrides)} residue override(s) will be applied "
          f"on top of global pH {GLOBAL_PH}.")
else:
    residue_overrides = None
    print(f"Tier 2 disabled: all residues will be titrated at global pH {GLOBAL_PH} only.")

# ==========================================
# 3. RUN BATCH CONVERSION (protonation + PDBQT)
# ==========================================
pdbqt_summary_df = batch_convert_protein_only(
    input_dir=INPUT_PDB_DIR,
    output_dir=OUTPUT_PDBQT_DIR,
    global_ph=GLOBAL_PH,
    residue_overrides=residue_overrides,
    audit_report_path=AUDIT_REPORT_PATH
)

# ==========================================
# 4. DISPLAY SUMMARY
# ==========================================
display(pdbqt_summary_df[[
    "filename", "global_ph", "custom_protonation_used", "residues_overridden",
    "residues_missing_override", "protonation_status", "mode", "status", "output_file"
]])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Tier 2 disabled: all residues will be titrated at global pH 7.4 only.
[14:29:54] [INFO] [Protein-Only Mode] Found 4 structure(s) to convert.
[14:30:07] [INFO] Protonated 'clean_5jmn.pdb' at pH 7.4 (0 fixed residue(s)) -> '_protonated_clean_5jmn.pdb'
[14:30:49] [INFO] Converted 'clean_5jmn.pdb' -> '5jmn.pdbqt'
[14:31:02] [INFO] Protonated 'clean_6zod.pdb' at pH 7.4 (0 fixed residue(s)) -> '_protonated_clean_6zod.pdb'
[14:31:03] [WARNING] Standard Meeko failed on 'clean_6zod.pdb'. Retrying with residue relaxation (--delete_bad_res)...
[14:31:04] [ERROR] Failed converting 'clean_6zod.pdb': rdkit.Chem.rdchem.AtomValenceException: Explicit valence for atom # 1 C, 5, is greater than permitted
[14:31:16] [INFO] Protonated 'clean_TM2-5jmn.pdb' at pH 7.4 (0 fixed residue(s)) -> '_protonated_clean_TM2-5jmn.pdb'
[14:32:00] [INFO] Converted 'clean_TM2-5jmn.pdb' -> 'TM2-5jmn.pdbqt'
[14:32:12] [INFO] Protonated '

,filename,global_ph,custom_protonation_used,residues_overridden,residues_missing_override,protonation_status,mode,status,output_file
0,clean_5jmn.pdb,7.4,False,0,[],Success,Protein-Only,Success,/home/amicon19/Docking/clean_TM-conf/5jmn.pdbqt
1,clean_6zod.pdb,7.4,False,0,[],Success,Protein-Only,Failed,/home/amicon19/Docking/clean_TM-conf/6zod.pdbqt
2,clean_TM2-5jmn.pdb,7.4,False,0,[],Success,Protein-Only,Success,/home/amicon19/Docking/clean_TM-conf/TM2-5jmn....
3,clean_TM3-6zod.pdb,7.4,False,0,[],Success,Protein-Only,Failed,/home/amicon19/Docking/clean_TM-conf/TM3-6zod....


In [ ]:
# Use this cell if your receptor contains cofactors, e.g. heme
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.receptor_pdbqt import batch_convert_with_cofactors

# ==========================================
# 1. PARAMETERS
# ==========================================
INPUT_PDB_DIR       = "./data/cleaned_ensemble"
OUTPUT_PDBQT_DIR    = "./data/receptor_pdbqt_heme"
AUDIT_REPORT_PATH   = "./data/pdbqt_heme_summary.csv"

# Optional: Path to custom Meeko JSON residue template if using novel cofactors
CUSTOM_TEMPLATES    = None  # e.g., "./data/custom_cofactors.json" or None

# ==========================================
# 2. RUN BATCH CONVERSION
# ==========================================
pdbqt_heme_df = batch_convert_with_cofactors(
    input_dir=INPUT_PDB_DIR,
    output_dir=OUTPUT_PDBQT_DIR,
    audit_report_path=AUDIT_REPORT_PATH,
    custom_template_path=CUSTOM_TEMPLATES
)

# ==========================================
# 3. DISPLAY SUMMARY
# ==========================================
display(pdbqt_heme_df[["filename", "mode", "status", "output_file"]])

## For docking protocol validation, prepare control ligand here

In [4]:
# Convert pdb to sdf and add explicit H - single molecule, for the co-crystallized control
from rdkit import Chem

# Define user-input filenames here
input_file = "ctrl_ligand/MIY.pdb"
output_file = "ctrl_lig4dock/MIY.sdf"

# Read the PDB file using the variable
mol = Chem.MolFromPDBFile(input_file, removeHs=False)

if mol is not None:
    # Add explicit hydrogens AND generate 3D coordinates for them
    mol_with_Hs = Chem.AddHs(mol, addCoords=True)
    
    # Write the molecule to an SDF file using the variable
    with Chem.SDWriter(output_file) as writer:
        writer.write(mol_with_Hs)
    print(f"Successfully added explicit hydrogens and converted to {output_file}!")
else:
    print(f"Error: RDKit could not read the PDB file '{input_file}'.")

Successfully added explicit hydrogens and converted to ctrl_lig4dock/MIY.sdf!


In [ ]:
# Convert mol2 to sdf, preserving the protonation states

In [12]:
!mk_prepare_ligand.py -i ctrl_lig4dock/RFP.sdf

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


In [6]:
!mk_prepare_ligand.py -i {output_file}

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


## Preparation of Ligand for docking

This script accepts one or more ligands in sdf format

### Conversion of SMILES to 3D sdf

This script takes an input of csv files with SMILES and ID column, and generates an sdf database of 3D structures. for unspecified stereoisomer configuration, it writes possible steroisomer variants of the molecule

In [1]:
#Convert a CSV (or folder of CSVs) with SMILES + ID columns into a single 3D SDF library.
#Protonation states encoded in the SMILES are preserved as-is (no pH adjustment).
#Unspecified stereocenters are enumerated into separate 3D stereoisomers, suffixed _1, _2, etc.

import pandas as pd
from src.csv_to_sdf_converter import batch_convert_csv_to_sdf

# ==========================================
# 1. PARAMETERS
# ==========================================
INPUT_SOURCE     = "TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv"                # a single CSV file OR a directory of CSVs
OUTPUT_SDF       = "TM1-pyrip0910_3d.sdf"       # combined 3D SDF output
LOG_REPORT_PATH  = "TM1-pyrip0910_3d_log.csv"     # per-molecule audit log

SMILES_COLUMN    = "SMILES"
ID_COLUMN        = "ID"
MAX_ISOMERS      = 32                               # safety cap per molecule

# ==========================================
# 2. RUN CONVERSION
# ==========================================
audit_df = batch_convert_csv_to_sdf(
    input_source=INPUT_SOURCE,
    output_sdf_path=OUTPUT_SDF,
    smiles_col=SMILES_COLUMN,
    id_col=ID_COLUMN,
    max_isomers=MAX_ISOMERS,
    log_report_path=LOG_REPORT_PATH
)

# ==========================================
# 3. DISPLAY AUDIT SUMMARY
# ==========================================
display(audit_df[[
    "source_file",
    "original_id",
    "num_stereoisomers_enumerated",
    "stereo_variation_applied",
    "output_ids",
    "status"
]])

[15:41:23] [INFO] Loaded 66 SMILES entries from 1 CSV file(s).
[15:41:25] [INFO] Audit log saved to '/home/amicon19/Docking/TM1-pyrip0910_3d_log.csv'.
[15:41:25] [INFO] ==================================================
[15:41:25] [INFO] CSV -> SDF CONVERSION SUMMARY
[15:41:25] [INFO] ==================================================
[15:41:25] [INFO] Input molecules processed        : 66
[15:41:25] [INFO] Molecules with stereo variants    : 5
[15:41:25] [INFO] Molecules with no stereo ambiguity: 61
[15:41:25] [INFO] Failed molecules                  : 0
[15:41:25] [INFO] Total 3D records written to SDF   : 72
[15:41:25] [INFO] Output SDF                        : /home/amicon19/Docking/TM1-pyrip0910_3d.sdf
[15:41:25] [INFO] ==================================================


,source_file,original_id,num_stereoisomers_enumerated,stereo_variation_applied,output_ids,status
0,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,Ple2022_1_BDM73185,1,No,Ple2022_1_BDM73185,Success
1,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,Ple2022_2,1,No,Ple2022_2,Success
2,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,Ple2022_3,1,No,Ple2022_3,Success
3,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,Ple2022_4,1,No,Ple2022_4,Success
4,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,Ple2022_5,1,No,Ple2022_5,Success
...,...,...,...,...,...,...
61,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,VDC2024_9,1,No,VDC2024_9,Success
62,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,VDC2024_10,1,No,VDC2024_10,Success
63,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,VDC2024_11,1,No,VDC2024_11,Success
64,TM-Pocket_PyrPip_SAR_Dataset_corr_v2.csv,VDC2024_12_3R,2,Yes,VDC2024_12_3R_1;VDC2024_12_3R_2,Success


### Converts the SDF database to pdbqt, preserving any protonation states

In [20]:
#Convert an SDF ligand library to individual PDBQT files.
#Preserves protonation states for named molecules and/or auto-detected charged states.
#Records that fail sanitization due to a fixable over-valent heteroatom (e.g. a
#protonated amine missing its '+' charge) are repaired instead of silently dropped.

import pandas as pd
from src.ligand_preparator import batch_convert_sdf_to_pdbqt

# ==========================================
# 1. PARAMETERS
# ==========================================
INPUT_SDF        = "ctrl_ligand/PABN.sdf"
OUTPUT_DIR       = "comparator_ligands/"
AUDIT_REPORT_PATH = "compare_PABN-ligand.csv"

TARGET_PH        = 7.0                       # ideal pH for Dimorphite-DL re-protonation
PRESERVE_NAMES   = set()                     # e.g. {"Ple2022_9_BDM88855HCl"}
AUTO_DETECT_CHARGED_STATES = True            # auto-preserve pre-charged molecules

# ==========================================
# 2. RUN BATCH CONVERSION
# ==========================================
audit_df = batch_convert_sdf_to_pdbqt(
    input_sdf=INPUT_SDF,
    output_dir=OUTPUT_DIR,
    target_ph=TARGET_PH,
    preserve_names=PRESERVE_NAMES,
    auto_detect_charged_states=AUTO_DETECT_CHARGED_STATES,
    audit_report_path=AUDIT_REPORT_PATH
)

# ==========================================
# 3. DISPLAY AUDIT SUMMARY
# ==========================================
display(audit_df[[
    "mol_name",
    "protonation_preserved",
    "repair_note",
    "status",
    "error_message"
]])

[15:10:50] [INFO] Successfully created: comparator_ligands/54592172.pdbqt
[15:10:50] [INFO] Audit log saved to '/home/amicon19/Docking/compare_PABN-ligand.csv'.
[15:10:50] [INFO] ==================================================
[15:10:50] [INFO] SDF -> PDBQT CONVERSION SUMMARY
[15:10:50] [INFO] ==================================================
[15:10:50] [INFO] Total records in SDF          : 1
[15:10:50] [INFO] Successfully written           : 1
[15:10:50] [INFO] Protonation preserved           : 0
[15:10:50] [INFO] Dropped (unfixable valence)     : 0
[15:10:50] [INFO] Failed (other errors)           : 0
[15:10:50] [INFO] Output directory                : /home/amicon19/Docking/comparator_ligands
[15:10:50] [INFO] ==================================================


/home/amicon19/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/preparation.py:693: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/home/amicon19/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/preparation.py:467: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


,mol_name,protonation_preserved,repair_note,status,error_message
0,54592172,No,,Success,


### ligand preparation from sdf at neutral pH

In [1]:
#Use this script if you already have a database of sdf structures and you are only extracting a subset of structures from a list

import pandas as pd
from src.sdf_extractor import load_query_ids, filter_sdf_by_id

# ==========================================
# 1. PARAMETERS FOR EXTRACTION
# ==========================================
QUERY_TABLE_PATH    = "ligands.csv"       # Delimited table of interest
QUERY_ID_COLUMN     = "HMDB_ID"                     # Target ID column header
SDF_DATABASE_PATH   = "/home/amicon19/Documents/EPI_dock/structures.sdf"   # Full database SDF
OUTPUT_SDF_PATH     = "seaweed_hmdb.sdf"                   # Output file fed into downstream script
MISSING_REPORT_PATH = "missing_hmdb.csv"  # Missing query IDs report

# Matches against <HMDB_ID>, then writes this ID to the first line (_Name) of each output record
SDF_ID_TAG          = ["HMDB_ID"]
SET_TITLE_LINE      = True                          # Sets line 1 of MolBlock to the matched ID

# ==========================================
# 2. RUN EXTRACTION
# ==========================================
query_ids = load_query_ids(
    file_path=QUERY_TABLE_PATH,
    id_column=QUERY_ID_COLUMN
)

results = filter_sdf_by_id(
    sdf_input_path=SDF_DATABASE_PATH,
    sdf_output_path=OUTPUT_SDF_PATH,
    query_ids=query_ids,
    sdf_id_tag=SDF_ID_TAG,
    set_name_to_matched_id=SET_TITLE_LINE,
    missing_report_path=MISSING_REPORT_PATH
)

[10:54:42] [INFO] Loaded 107 unique query ID(s) from 'ligands.csv'.
[10:54:42] [INFO] Starting scan of 'structures.sdf' matching against property tag(s) ['HMDB_ID']...


[10:54:42] The 2 defining bonds for an atropisomer are co-planar - atoms are: 27 28
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 12 ignored
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 12 ignored
[10:54:42] The 2 defining bonds for an atropisomer are co-planar - atoms are: 16 27
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 3 ignored
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 3 ignored
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 29 ignored
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 29 ignored


[10:54:42] [INFO] Progress: 10,000 scanned | 6 / 107 IDs matched


[10:54:42] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 131 ignored.
[10:54:42] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 131 ignored.
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 1 ignored
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 1 ignored
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 7 ignored
[10:54:42] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 7 ignored
[10:54:42] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 20 ignored.
[10:54:42] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 20 ignored.


[10:54:43] [INFO] Progress: 20,000 scanned | 65 / 107 IDs matched


...

[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 70 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 69 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 69 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 69 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 69 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 60 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 60 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 70 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 69 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 69 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final c

[10:54:54] [INFO] Progress: 210,000 scanned | 90 / 107 IDs matched


[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 70 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 70 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 70 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 72 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 71 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 71 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 71 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 71 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 73 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 72 ignored
[10:54:54] Warning: ambiguous stereochemistry - zero final c

[10:54:54] [INFO] =============================================
[10:54:54] [INFO] EXTRACTION SUMMARY
[10:54:54] [INFO] =============================================
[10:54:54] [INFO] Total entries scanned     : 217,776
[10:54:54] [INFO] Corrupted entries skipped : 1
[10:54:54] [INFO] Molecules exported        : 107
[10:54:54] [INFO] Unique IDs matched        : 107 / 107 (100.00%)
[10:54:54] [INFO] Unique IDs missing        : 0
[10:54:54] [INFO] Extracted SDF location    : /home/amicon19/Docking/seaweed_hmdb.sdf
[10:54:54] [INFO] =============================================


[10:54:54] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 21 ignored
[10:54:54] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 21 ignored
[10:54:54] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 21 ignored
[10:54:54] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 21 ignored
[10:54:54] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 13 ignored.
[10:54:54] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 13 ignored.
[10:54:54] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 10 ignored
[10:54:54] Warning: conflicting stereochemistry - bond wedging contradiction - at atom 10 ignored


In [3]:
#convert sdf library to individual pdbqt files

import os
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation
from dimorphite_dl import protonate_smiles  # Corrected Import

# 1. Define your file paths
input_sdf = "TM-PyrPip_3d.sdf"       
output_dir = "TM-PyrPip_ligand"     
target_ph = 7.0  #ideal pH 

os.makedirs(output_dir, exist_ok=True)
supplier = Chem.SDMolSupplier(input_sdf, removeHs=False)
mk_prep = MoleculePreparation()

for mol in supplier:
    if mol is None:
        continue
        
    mol_name = mol.GetProp('_Name').strip() if (mol.HasProp('_Name') and mol.GetProp('_Name').strip()) else f"ligand_{mol.GetNumAtoms()}_atoms"

    try:
        # Convert RDKit mol to SMILES for Dimorphite-DL
        original_smiles = Chem.MolToSmiles(mol)
        
        # 2. Fix Protonation State at Physiological pH
        # Returns a list of protonated SMILES strings
        protonated_smiles_list = protonate_smiles(
            original_smiles, 
            ph_min=target_ph - 0.5, 
            ph_max=target_ph + 0.5
        )
        
        if not protonated_smiles_list:
            print(f"Skipping {mol_name}: Could not determine protonation state.")
            continue
            
        # Take the first predicted protonation state
        best_smiles = protonated_smiles_list[0]
        
        # Convert back to RDKit molecule
        best_mol = Chem.MolFromSmiles(best_smiles)
        
        # 3. Add explicit hydrogens corresponding to the new protonation state
        best_mol = Chem.AddHs(best_mol)
        
        # 4. Generate a new 3D conformation
        embed_status = AllChem.EmbedMolecule(best_mol, AllChem.ETKDGv3())
        if embed_status != 0:
             print(f"Warning: Conformer generation failed for {mol_name}. Proceeding with fallback.")
             AllChem.Compute2DCoords(best_mol) 
        
        AllChem.MMFFOptimizeMolecule(best_mol)

        # 5. Process into PDBQT format
        mk_prep.prepare(best_mol)
        pdbqt_string = mk_prep.write_pdbqt_string()
        
        output_filepath = os.path.join(output_dir, f"{mol_name}.pdbqt")
        with open(output_filepath, "w") as f:
            f.write(pdbqt_string)
            
        print(f"Successfully created: {output_filepath}")
        
    except Exception as e:
        print(f"Error processing {mol_name}: {e}")

/home/amicon19/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/preparation.py:693: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/home/amicon19/miniconda3/envs/vina/lib/python3.11/site-packages/meeko/preparation.py:467: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
[RDKit] ERROR:[14:16:11] Explicit valence for atom # 1 N, 4, is greater than permitted
[RDKit] ERROR:[14:16:11] Explicit valence for atom # 1 N, 4, is greater than permitted
[RDKit] ERROR:[14:16:11] Explicit valence for atom # 6 N, 4, is greater than permitted
[RDKit] ERROR:[14:16:11] Explicit valence for atom # 6 N, 4, is greater than permitted
[RDKit] 

Successfully created: TM-PyrPip_ligand/Ple2022_1_BDM73185.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_2.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_3.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_4.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_5.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_6.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_7.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_8_BDM88832.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_9_BDM88855HCl.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_10.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_11.pdbqt
Successfully created: TM-PyrPip_ligand/Ple2022_12.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_1_BDM73185.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_2_BDM88832.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_3.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_4_1.pdbqt


[RDKit] ERROR:[14:16:11] Explicit valence for atom # 6 N, 4, is greater than permitted
[RDKit] ERROR:[14:16:11] Explicit valence for atom # 6 N, 4, is greater than permitted
[RDKit] ERROR:[14:16:11] Explicit valence for atom # 6 N, 4, is greater than permitted
[RDKit] ERROR:[14:16:11] Explicit valence for atom # 6 N, 4, is greater than permitted


Successfully created: TM-PyrPip_ligand/Compagne2023_4_2.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_5_1.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_5_2.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_6_1.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_6_2.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_6_3.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_7.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_8.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_9.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_10.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_11.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_12.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_13.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_14.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_15.pdbqt
Successfully created: TM-PyrPip_ligand/Compagne2023_16.pdbqt
Successfully created:

## Run docking screen
Make sure to edit input and output names and parameters

In [8]:
import pandas as pd
from src.vinascreen_runner import run_vinascreen_batch

# =========================================================
# 1. BATCH DIRECTORIES & OUTPUT PATHS (Retained)
# =========================================================
BATCH_NAME       = "P23-validate"
RECEPTOR_DIR     = "./L-receptors4dock/"
LIGAND_DIR       = "./ctrl_ligand/"
OUTPUT_POSE_DIR  = f"./results/{BATCH_NAME}/poses"
RESULTS_CSV_PATH = f"./results/{BATCH_NAME}/docking_scores.csv"

# Optional: specify binary path if not in system PATH (e.g. "./src/vina_1.2.7_linux_x86_64")
VINA_EXECUTABLE  = None 

# =========================================================
# 2. DOCKING PARAMETERS (Retained)
# =========================================================
DOCKING_PARAMS = {
    # Binding Pocket Center Coordinates (X, Y, Z)
    "center_x": -66.96,
    "center_y": -25.05,
    "center_z": -66.69,
    
    # Grid Box Dimensions (Angstroms)
    "size_x": 15.0,
    "size_y": 15.0,
    "size_z": 15.0,
    
    # Search Settings
    "exhaustiveness": 16,
    "num_modes": 10,
    "energy_range": 3,
    "cpu": 4,        # 4 thread/core per Vina task instance
    "seed": 42       # Ensures reproducibility across runs
}

# =========================================================
# 3. RUN BATCH SCREENING (Updated: max_workers added)
# =========================================================
results_df = run_vinascreen_batch(
    receptor_dir=RECEPTOR_DIR,
    ligand_dir=LIGAND_DIR,
    output_dir=OUTPUT_POSE_DIR,
    docking_config=DOCKING_PARAMS,
    results_csv=RESULTS_CSV_PATH,
    vina_executable=VINA_EXECUTABLE,
    max_workers=5   # <--- 5 workers * 4 cpu cores = Uses 20 of your 24 CPU cores concurrently
)

# =========================================================
# 4. PREVIEW TOP HITS (Retained)
# =========================================================
print("\nTop 10 High-Affinity Hits:")
top_hits = results_df[results_df["status"] == "Success"].sort_values(by="affinity_kcal_mol").head(10)
display(top_hits[["receptor", "ligand", "affinity_kcal_mol", "rmsd_lb", "status"]])

[11:27:41] [INFO] Using AutoDock Vina binary: '/home/amicon19/Docking/src/vina_1.2.7_linux_x86_64'
[11:27:41] [INFO] Initialized VinaScreen: 9 receptor(s) x 2 ligand(s) = 18 total docking jobs across 5 parallel workers.
[11:48:49] [INFO] Progress: [18/18] jobs completed.
[11:48:49] [INFO] Results table successfully written to '/home/amicon19/Docking/results/P23-validate/docking_scores.csv'.
[11:48:49] [INFO] =============================================
[11:48:49] [INFO] VINASCREEN DOCKING COMPLETED
[11:48:49] [INFO] =============================================
[11:48:49] [INFO] Total jobs executed : 18
[11:48:49] [INFO] Successful runs     : 9
[11:48:49] [INFO] Failed runs         : 9
[11:48:49] [INFO] Output directory    : /home/amicon19/Docking/results/P23-validate/poses
[11:48:49] [INFO] =============================================

Top 10 High-Affinity Hits:


,receptor,ligand,affinity_kcal_mol,rmsd_lb,status
16,L-7oul,MIY,-7.827,0.0,Success
6,L-3aod,MIY,-7.664,0.0,Success
2,L-3aob,MIY,-7.135,0.0,Success
10,L-5enq-a,MIY,-7.103,0.0,Success
14,L-7ouk,MIY,-7.041,0.0,Success
8,L-3w9h,MIY,-7.029,0.0,Success
4,L-3aoc,MIY,-6.795,0.0,Success
0,L-1iwg,MIY,-6.673,0.0,Success
12,L-5jmn,MIY,-6.423,0.0,Success


## Evaluation of Docking Control

In [7]:
# Validation of control — Jupyter front-end
import sys, os
sys.path.append(os.path.join(os.getcwd(), "src"))  # adjust if your src folder lives elsewhere
from docking_validation import validate_docking

# ==========================================
#               USER SETTINGS
# ==========================================
docked_pdbqt_file      = "results/D2_qsar_MBX911/poses/D-5enr_MBX3135_docked.pdbqt"
crystal_reference_mol2 = "ctrl_ligand/MBX3135.mol2"

temp_sdf_file    = "results/D2_qsar_MBX911/D5enr-mbx3135.sdf"
text_report_file = "results/D2_qsar_MBX911/D5enr-mbx3135.txt"

rmsd_cutoff = 2.0
# ==========================================

result = validate_docking(
    docked_pdbqt_file=docked_pdbqt_file,
    crystal_reference_mol2=crystal_reference_mol2,
    temp_sdf_file=temp_sdf_file,
    text_report_file=text_report_file,
    rmsd_cutoff=rmsd_cutoff,
)

[12:17:23] 5enr: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.



  DOCKING PROTOCOL VALIDATION REPORT
  Pose     RMSD (A)       Matched    Result
-------------------------------------------------------
  Pose 1    0.974          36/36      PASS  <==
  Pose 2    2.158          36/36      FAIL
  Pose 3    10.105         36/36      FAIL
  Pose 4    8.148          36/36      FAIL
  Pose 5    9.807          36/36      FAIL
  Pose 6    7.227          36/36      FAIL
  Pose 7    4.775          36/36      FAIL
  Pose 8    8.554          36/36      FAIL
  Pose 9    4.830          36/36      FAIL
  Pose 10   8.392          36/36      FAIL
  Best pose  : Pose 1 (0.974 A)
  Poses pass : 1 / 10
  Verdict    : PROTOCOL VALIDATED


# Convert VinaScreen pdbqt outputs

In [2]:
import os
import re
import pandas as pd
from openbabel import pybel

def parse_vina_pdbqt(filename):
    """Extract models and scores from vina PDBQT output file."""
    results = []
    with open(filename, 'r') as f:
        content = f.read()
    # Split models: each model starts with "MODEL" and ends with "ENDMDL"
    models = re.split('(MODEL.*?ENDMDL)', content, flags=re.DOTALL)
    model_number = 1
    for model in models:
        if 'MODEL' in model and 'ENDMDL' in model:
            # Extract vina score (affinity) from REMARK line
            score_match = re.search(r'REMARK VINA RESULT:\s*([-.\d]+)', model)
            if score_match:
                score = float(score_match.group(1))
            else:
                score = None
            # Write each pose to temp pdbqt for conversion
            temp_name = '_temp_model.pdbqt'
            with open(temp_name, 'w') as out:
                out.write(model)
            # Use openbabel to convert to SMILES
            mols = list(pybel.readfile("pdbqt", temp_name))
            if mols:
                smiles = mols[0].write(format="smi").split()[0]
            else:
                smiles = None
            os.remove(temp_name)
            results.append((model_number, score, smiles))
            model_number += 1
    return results

data = []
folder = "output_PDL1G1" #path to folder
for fname in os.listdir(folder):
    if fname.endswith(".pdbqt"):
        full_path = os.path.join(folder, fname)
        basename = os.path.splitext(fname)[0]
        models = parse_vina_pdbqt(full_path)
        for model_id, vina_score, smiles in models:
            row_id = f"{basename}_{model_id}"
            data.append({'ID': row_id, 'SMILES': smiles, 'vina_score': vina_score})

# Output as CSV
df = pd.DataFrame(data)

# Sort to ensure lowest model number comes first for each SMILES
df['model_num'] = df['ID'].str.extract(r'_(\d+)$').astype(int)
df.sort_values(by=['SMILES', 'model_num'], inplace=True)
# Drop duplicates keeping lowest model number (first occurrence)
df_no_dupes = df.drop_duplicates(subset=["SMILES"], keep='first')
# Drop auxiliary column before saving
df_no_dupes = df_no_dupes.drop(columns=['model_num'])
df_no_dupes.to_csv("vina_PDL1G1.csv", index=False) #change file name


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0250)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0250)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0250)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0320)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0320)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0320)

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is pdL1G1_PdL1G1_0202)

*** Op